<a href="https://colab.research.google.com/github/Vitorhugofsousa/pharmaNE/blob/main/pharmane.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Study case from Pharma Nordeste



In [ ]:
#Importing libs and mounting drive
import pandas as pd
import numpy as np
import os
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

In [ ]:
#Importing tables csv files
df_pharmacies = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/tabelas_pharma/dim_farmacias_final.csv')
df_medicamentlst = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/tabelas_pharma/lista_medicamentos_expandida.csv')
df_pharmacies_cap = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/tabelas_pharma/dim_capacidade_farmacia.csv')
df_batchs = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/tabelas_pharma/dim_lotes_validade.csv')
df_distribuition = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/tabelas_pharma/distribuicao_medicamentos_completa.csv')
df_shipments = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/tabelas_pharma/historico_remessas_2sem.csv')
df_logistical_occ = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/tabelas_pharma/ocorrencias_logisticas.csv')
df_sales = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/tabelas_pharma/tabela_vendas.csv')
df_clients = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/tabelas_pharma/dim_clientes.csv')

## Searching and founding problems in this dataset

In [ ]:
# Creating a function to analyse all the standard types of errors, it will be used in all these tables to find common errors.
def standard_analysis(df, table_name):
  print(f"--- Analising: {table_name} ---")
  print(f"Rows: {df.shape[0]} & Columns: {df.shape[1]}")
  print("\nTypes from data and Null's:")
  print(df.info())
  print(df.isnull().sum()) # Searching the total null values per column
  print("\nTotal Duplicates:")
  print(df.duplicated().sum()) # Searching for duplicated Values
  print("-" * 30)

In [ ]:
# dim_pharmacies analysis
standard_analysis(df_pharmacies, df_pharmacies) #aplying the standard function

df_pharmacies['cod_farmacia'].is_unique #If was true, the pharmacies codes all total uniques, without problems with duplicantions
df_pharmacies['cidade'].unique() #--- all city names are padronized ---
df_pharmacies['cep'].astype(str).str.len().value_counts() # verify if all postal codes are standardized with the correct format
display(df_pharmacies.head(5))

In [ ]:
#df_meddicamentlst analysis
standard_analysis(df_medicamentlst, df_medicamentlst) #aplying the standard funcion to this table

df_medicamentlst[df_medicamentlst['preco_unitario'] <= 0] # verifying if the unitary price of sell isnt 0
df_medicamentlst[df_medicamentlst['preco_custo_unitario'] <= 0] #verifiyng if the unitary price of the distribuitor isnt 0
df_medicamentlst['categoria'].value_counts() # verifying the total of medications per category
df_medicamentlst.duplicated(subset=['nome_medicamento', 'dosagem', 'fabricante']).sum() #  verifying duplicated items in the medicaments list by a combination of columns
display(df_medicamentlst.head(5))

In [ ]:
#df_pharmacies_cap analysis
standard_analysis(df_pharmacies_cap, df_pharmacies_cap) # aplying the standard analysis funcion to this table

df_pharmacies_cap[['area_total_m2', 'capacidade_geladeira_litros']].describe() # verifying if the areas and capacities are compatible with the pharmacies and medicaments or if have any issue with the informations of the capacities

In [ ]:
#df_batchs analysis
standard_analysis(df_batchs, df_batchs) # aplying the standard analysis function to this table

df_batchs['data_fabricacao'] = pd.to_datetime(df_batchs['data_fabricacao'])
df_batchs['data_validade'] = pd.to_datetime(df_batchs['data_validade'])
# Converting the order of the batchs per date

error_dates = df_batchs[df_batchs['data_fabricacao'] > df_batchs['data_validade']]
print(len(error_dates)) # verifying if there are errors in the informations of dates from fabrication

In [ ]:
#df_distribuition analysis
standard_analysis(df_distribuition, df_distribuition) # aplying the standard analysis function to this table

valid_ids = df_pharmacies['cod_farmacia'].unique()
errors_fk = df_distribuition[~df_distribuition['cod_farmacia'].isin(valid_ids)]
print(f"Pharmacies who doesnt exists on the registry {len(errors_fk)}")
# verifying if has erros from foreign key what represents the id of all pharmacies

estoque_por_farmacia = df_distribuition.groupby('cod_farmacia')['quantidade_estoque'].sum()
display(estoque_por_farmacia)

# ERRORS FOUNDED: NULL VALUES AT COLUMNS - (substancia_princial, apresentacao, fabricante, nome_farmacia, cidade)

In [ ]:
#df_shipments analysis
standard_analysis(df_shipments, df_shipments) # aplying the standard analysis function to this table

valid_batchs = df_batchs['lote_fabricacao'].unique()
errors_fk_fab = df_shipments[~df_shipments['lote_fabricacao'].isin(valid_batchs)]
print(f"Shipments who doesnt exists on the registry {len(errors_fk_fab)}")

quantidade_total_med = df_shipments.groupby('cod_farmacia')['quantidade_recebida'].sum().reset_index().sort_values(by='quantidade_recebida', ascending=False)
display(quantidade_total_med)

#ERRORS FOUNDED: NULL VALUES AT COLUMN - (data_entrega)

In [ ]:
#df_logistical_occ analysis
standard_analysis(df_logistical_occ, df_logistical_occ) # aplying the standard analysis function to this table

df_logistical_occ['tipo_movimento'].value_counts() # Verifying if troubles with logistical shipment are standardized

In [ ]:
#df_sales analysis
standard_analysis(df_sales, df_sales) # aplying the standard analysis function to this table

df_sales['data_venda'] = pd.to_datetime(df_sales['data_venda']) # changing the date order of sells
df_sales[df_sales['valor_total_compra'] <= 0] # verifying errors in the total amount of a puchase
df_sales[df_sales['qtd_produtos'] <= 0] # verifying errors in the number of products sold in a puchase
df_sales['metodo_pagamento'].value_counts() #Display the total sales by perpayment method
df_sales['id_cliente'].value_counts().reset_index().sort_values(by='count', ascending=False) # Display the total sales by registred buyer
df_sales['cod_farmacia'].value_counts().reset_index().sort_values(by='count', ascending=False) # Display the total of sales by estabilishment
df_sales['cod_medicamento'].value_counts()


#ERRORS FOUNDED: DUPLICATED ROWS - AND - NULL VALUES AT COLUMN (metodo_pagamento)

In [ ]:
#df_clients analysis
standard_analysis(df_clients, df_clients) # Aplying the standard analysis to this table

### **Conclusion and next steps**
#### All the Problems Founded in this dataset

At the df where is the informacion of the location of the estabilishment *df_pharmacies* we didnt find any problems with duplicated values, empty cells or non standardized values.

At the dataframe where is the information of all the registred medicaments *df_medicamentlst* we didnt find any problems with duplicated values, empty cells or non standardized values too.

At the dataframe where is the information of the estabilishment capacities of medicaments stock *df_pharmacies_cap* we didnt find any problems with duplicated values, empty cells or non standardized values too.

At thw dataframe where is the information of the fabrication batchs of the medicaments existing on the dataset *df_batchs* we didnt find any problems with duplicated values, empty cells or non standardized values too.

At the dataframe where is the information of the distribuition of medicaments in all pharmacies *df_distribuition* we founded the following errors NULL VALUES AT COLUMNS - (substancia_princial, apresentacao, fabricante, nome_farmacia, cidade)

At the dataframe of realized medicament shipments *df_shipments* we find the following errors NULL VALUES AT COLUMN - (data_entrega)

At the dataframe where shows the logistical issues with the medicament shipments *df_logistical_occ* e didnt find any problems with duplicated values, empty cells or non standardized values too.

At the dataframe where shows the total of sales *df_sales* we founded the following errors DUPLICATED ROWS - AND - NULL VALUES AT COLUMN (metodo_pagamento).

At the dataflame where we find the informations about the registred clients *df_clients* we didnt find any problems.
<br><br>

### **Next Steps**
Now I will normalize the tables and fill the cells with empty values with the correct values.

Later I intend make optimizations with the tables in this database, for example I think is a better idea join the tables pharmacies and pharmacie capacities, because both of tables shows descriptive informations about all pharmacies. Join these tables and delete unnecessary columns will make a lot of difference.
Some columns in different tables are duplicated too, maybe we will need to replace some of the informacion just for ids to reduce redundancy.

Now i will continue the next steps in another notebook especifically to normalize the data, then i will export these archives to upload to an SQL database.